In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls /content/drive


MyDrive


In [ ]:
!ls /content/drive/MyDrive/oil_spill_project


data  metadata


In [ ]:
!ls /content/drive/MyDrive/oil_spill_project/data


kaggle	kaggle_raw  sentinel


In [ ]:
!ls /content/drive/MyDrive/oil_spill_project/data/kaggle_raw


S1SAR_UnBalanced_400by400_Class_0  S1SAR_UnBalanced_400by400_Class_1


In [ ]:
!ls /content/drive/MyDrive/oil_spill_project/data/kaggle_raw/S1SAR_UnBalanced_400by400_Class_0


0


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

RAW_BASE = "/content/drive/MyDrive/oil_spill_project/data/kaggle_raw"
OUT_BASE = "/content/drive/MyDrive/oil_spill_project/data/kaggle"

# Define exact source paths
sources = {
    "0": os.path.join(RAW_BASE, "S1SAR_UnBalanced_400by400_Class_0", "0"),
    "1": os.path.join(RAW_BASE, "S1SAR_UnBalanced_400by400_Class_1", "1"),
}

# Ensure output folders exist
for split in ["train", "val"]:
    for label in ["0", "1"]:
        os.makedirs(os.path.join(OUT_BASE, split, label), exist_ok=True)

# Split and copy
for label, src_dir in sources.items():
    images = [
        f for f in os.listdir(src_dir)
        if f.lower().endswith((".jpg", ".png", ".tif"))
    ]

    train_imgs, val_imgs = train_test_split(
        images, test_size=0.2, random_state=42
    )

    for img in train_imgs:
        shutil.copy(
            os.path.join(src_dir, img),
            os.path.join(OUT_BASE, "train", label, img)
        )

    for img in val_imgs:
        shutil.copy(
            os.path.join(src_dir, img),
            os.path.join(OUT_BASE, "val", label, img)
        )

print("✅ Kaggle dataset successfully split into train and val.")


✅ Kaggle dataset successfully split into train and val.


In [ ]:
print("Train 0:", len(os.listdir("/content/drive/MyDrive/oil_spill_project/data/kaggle/train/0")))
print("Train 1:", len(os.listdir("/content/drive/MyDrive/oil_spill_project/data/kaggle/train/1")))
print("Val 0:", len(os.listdir("/content/drive/MyDrive/oil_spill_project/data/kaggle/val/0")))
print("Val 1:", len(os.listdir("/content/drive/MyDrive/oil_spill_project/data/kaggle/val/1")))


Train 0: 2982
Train 1: 1524
Val 0: 746
Val 1: 381


In [ ]:
!pip install rasterio numpy opencv-python tqdm


In [ ]:
import os
import numpy as np
import rasterio
import cv2
from tqdm import tqdm

def tile_sentinel_image(
    tif_path,
    out_dir,
    tile_size=400,
    stride=400,
    prefix="tile"
):
    os.makedirs(out_dir, exist_ok=True)

    with rasterio.open(tif_path) as src:
        img = src.read(1)  # single-band SAR
        height, width = img.shape

    tile_count = 0

    for y in range(0, height - tile_size + 1, stride):
        for x in range(0, width - tile_size + 1, stride):

            tile = img[y:y+tile_size, x:x+tile_size]

            # Skip empty tiles (all zeros or NaNs)
            if np.all(tile == 0) or np.isnan(tile).all():
                continue

            # Normalize tile to 0–255 for visibility
            tile_norm = cv2.normalize(
                tile, None, 0, 255, cv2.NORM_MINMAX
            ).astype(np.uint8)

            tile_name = f"{prefix}_{tile_count:05d}.png"
            cv2.imwrite(os.path.join(out_dir, tile_name), tile_norm)

            tile_count += 1

    print(f"✅ {tile_count} tiles saved to {out_dir}")


In [ ]:
tile_sentinel_image(
    tif_path="/content/drive/MyDrive/oil_spill_project/data/sentinel/raw/aug09.tif",
    out_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug09/images",
    tile_size=400,
    stride=400,
    prefix="aug09"
)


✅ 3208 tiles saved to /content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug09/images


In [ ]:
tile_sentinel_image(
    tif_path="/content/drive/MyDrive/oil_spill_project/data/sentinel/raw/aug12.tif",
    out_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug12/images",
    tile_size=400,
    stride=400,
    prefix="aug12"
)


✅ 3154 tiles saved to /content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug12/images


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import shutil

def filter_tiles(
    in_dir,
    out_dir,
    variance_thresh=5.0,
    land_pixel_thresh=200,
    max_land_percent=30
):
    os.makedirs(out_dir, exist_ok=True)

    kept, dropped_blank, dropped_land = 0, 0, 0

    for fname in tqdm(os.listdir(in_dir)):
        if not fname.endswith(".png"):
            continue

        img_path = os.path.join(in_dir, fname)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        # 1️⃣ Blank tile check (variance)
        if np.var(img) < variance_thresh:
            dropped_blank += 1
            continue

        # 2️⃣ Land percentage check
        land_pixels = np.sum(img > land_pixel_thresh)
        land_percent = (land_pixels / img.size) * 100

        if land_percent > max_land_percent:
            dropped_land += 1
            continue

        # Keep tile
        shutil.copy(img_path, os.path.join(out_dir, fname))
        kept += 1

    print("✅ Filtering complete")
    print(f"Kept tiles: {kept}")
    print(f"Dropped blank tiles: {dropped_blank}")
    print(f"Dropped land-dominated tiles: {dropped_land}")


In [ ]:
filter_tiles(
    in_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug09/images",
    out_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug09/filtered"
)


100%|██████████| 3208/3208 [01:33<00:00, 34.13it/s]

✅ Filtering complete
Kept tiles: 2918
Dropped blank tiles: 0
Dropped land-dominated tiles: 290


In [ ]:
filter_tiles(
    in_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug12/images",
    out_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug12/filtered"
)


100%|██████████| 3154/3154 [01:28<00:00, 35.70it/s]

✅ Filtering complete
Kept tiles: 2952
Dropped blank tiles: 3
Dropped land-dominated tiles: 199


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import shutil

def filter_tiles_strict(
    in_dir,
    out_dir,
    variance_thresh=10.0,
    land_pixel_thresh=200,
    max_land_percent=25
):
    os.makedirs(out_dir, exist_ok=True)

    kept, dropped_blank, dropped_land = 0, 0, 0

    for fname in tqdm(os.listdir(in_dir)):
        if not fname.endswith(".png"):
            continue

        img_path = os.path.join(in_dir, fname)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        # 1️⃣ Blank / low-information rejection
        if np.var(img) < variance_thresh:
            dropped_blank += 1
            continue

        # 2️⃣ Land-percentage rejection
        land_pixels = np.sum(img > land_pixel_thresh)
        land_percent = (land_pixels / img.size) * 100

        if land_percent > max_land_percent:
            dropped_land += 1
            continue

        shutil.copy(img_path, os.path.join(out_dir, fname))
        kept += 1

    print("✅ STRICT filtering complete")
    print(f"Kept tiles: {kept}")
    print(f"Dropped blank tiles: {dropped_blank}")
    print(f"Dropped land-dominated tiles: {dropped_land}")


In [ ]:
filter_tiles_strict(
    in_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug09/images",
    out_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug09/filtered_strict"
)


100%|██████████| 3208/3208 [01:40<00:00, 31.96it/s]


✅ STRICT filtering complete
Kept tiles: 2849
Dropped blank tiles: 0
Dropped land-dominated tiles: 359


In [ ]:
filter_tiles_strict(
    in_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug12/images",
    out_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug12/filtered_strict"
)


100%|██████████| 3154/3154 [01:33<00:00, 33.80it/s]

✅ STRICT filtering complete
Kept tiles: 2924
Dropped blank tiles: 3
Dropped land-dominated tiles: 227


In [ ]:
import os
import shutil
from tqdm import tqdm

src_dirs = [
    "/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug09/filtered_strict",
    "/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/aug12/filtered_strict",
]

pooled_dir = "/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/pooled/images"
os.makedirs(pooled_dir, exist_ok=True)

count = 0
for src in src_dirs:
    for fname in tqdm(os.listdir(src)):
        if fname.endswith(".png"):
            shutil.copy(
                os.path.join(src, fname),
                os.path.join(pooled_dir, fname)
            )
            count += 1

print(f"✅ Pooled {count} tiles into pooled/images")


100%|██████████| 2924/2924 [01:07<00:00, 43.38it/s]

✅ Pooled 5773 tiles into pooled/images


In [ ]:
import random
import shutil
import os

pooled_dir = "/content/drive/MyDrive/oil_spill_project/data/sentinel/tile_pool/pooled/images"

out_base = "/content/drive/MyDrive/oil_spill_project/data/sentinel/tiles"

splits = {
    "train": 0.70,
    "val": 0.15,
    "primary_test": 0.15
}

# Create output folders
for split in splits:
    os.makedirs(os.path.join(out_base, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(out_base, split, "masks"), exist_ok=True)

files = [f for f in os.listdir(pooled_dir) if f.endswith(".png")]
random.seed(42)
random.shuffle(files)

n_total = len(files)
n_train = int(n_total * splits["train"])
n_val = int(n_total * splits["val"])

train_files = files[:n_train]
val_files = files[n_train:n_train + n_val]
test_files = files[n_train + n_val:]

def copy_files(file_list, split_name):
    for f in file_list:
        shutil.copy(
            os.path.join(pooled_dir, f),
            os.path.join(out_base, split_name, "images", f)
        )

copy_files(train_files, "train")
copy_files(val_files, "val")
copy_files(test_files, "primary_test")

print("✅ Sentinel tiles split completed")
print(f"Train: {len(train_files)}")
print(f"Val: {len(val_files)}")
print(f"Primary Test: {len(test_files)}")


✅ Sentinel tiles split completed
Train: 4041
Val: 865
Primary Test: 867


In [ ]:
import os
import numpy as np
import rasterio
import cv2
from tqdm import tqdm

def tile_stress_test_final(
    tif_path,
    out_dir,
    tile_size=400,
    stride=400,            # ✅ NO OVERLAP
    prefix="aug05",
    ocean_threshold=180,
    min_ocean_percent=30
):
    os.makedirs(out_dir, exist_ok=True)

    with rasterio.open(tif_path) as src:
        img = src.read(1)
        height, width = img.shape

    tile_count = 0
    skipped_empty = 0
    skipped_land = 0

    for y in tqdm(range(0, height - tile_size + 1, stride)):
        for x in range(0, width - tile_size + 1, stride):

            tile = img[y:y+tile_size, x:x+tile_size]

            # Drop empty / nodata tiles
            if np.all(tile == 0) or np.isnan(tile).all():
                skipped_empty += 1
                continue

            tile_norm = cv2.normalize(
                tile, None, 0, 255, cv2.NORM_MINMAX
            ).astype(np.uint8)

            # Ocean presence check
            ocean_pixels = np.sum(tile_norm < ocean_threshold)
            ocean_percent = (ocean_pixels / tile_norm.size) * 100

            # Drop land-dominated tiles
            if ocean_percent < min_ocean_percent:
                skipped_land += 1
                continue

            tile_name = f"{prefix}_{tile_count:05d}.png"
            cv2.imwrite(os.path.join(out_dir, tile_name), tile_norm)
            tile_count += 1

    print("✅ Stress-test tiling completed (FINAL, no overlap)")
    print(f"Tiles saved: {tile_count}")
    print(f"Empty tiles skipped: {skipped_empty}")
    print(f"Land-dominated tiles skipped: {skipped_land}")


In [ ]:
tile_stress_test_final(
    tif_path="/content/drive/MyDrive/oil_spill_project/data/sentinel/raw/aug05.tif",
    out_dir="/content/drive/MyDrive/oil_spill_project/data/sentinel/stress_test/aug05/images",
    tile_size=400,
    stride=400,   # explicit
    prefix="aug05"
)


100%|██████████| 51/51 [00:41<00:00,  1.24it/s]

✅ Stress-test tiling completed (FINAL, no overlap)
Tiles saved: 2712
Empty tiles skipped: 1143
Land-dominated tiles skipped: 174
